# CARLA plant check on Colab (experiment E13)
**UNTESTED.** Written without being able to run it. Expect to fix a few cells. The commands, URLs and versions below are
best guesses: check them against the official CARLA release page. Every cell says what could break.

Goal: run `experiments/e13_carla_plant.py` (does the closed-form braking plant agree with CARLA's physics?) and download
`e13_carla.json`. Use a **GPU runtime** (Runtime > Change runtime type > T4 GPU). If this is not working after about an
hour, stop: the paper can list the CARLA check as future work.

## 0. Make a small project zip on your laptop (not in Colab)
In the project folder run:
```
git archive -o sdv_min.zip HEAD sdv config.py config.yaml experiments/e13_carla_plant.py Knowledge_base requirements.txt
```
Then upload `sdv_min.zip` in the next cell.

In [ ]:
!nvidia-smi
import sys; print(sys.version)

In [ ]:
from google.colab import files
up = files.upload()            # choose sdv_min.zip
!unzip -q -o sdv_min.zip -d /content/proj && ls /content/proj

## 1. Download the CARLA server
The file is about 8.4 GB (a few minutes on Colab). If the assert fails, the download did not complete: delete the file and run the cell again.


In [ ]:
import os
URL = "https://downloads.carlasim.com/Linux/CARLA_0.9.15.tar.gz"   # official host; the old carla-releases.b-cdn.net link now returns 403 (checked 2026-10-07)
!df -h /content | tail -1
!wget -nv --tries=3 {URL} -O /content/carla.tar.gz
size = os.path.getsize('/content/carla.tar.gz')
print(f'downloaded {size / 1e9:.2f} GB (expected about 8.39 GB)')
assert size > 8.3e9, 'download incomplete or an error page was saved; do not untar it'
!mkdir -p /content/carla && tar -xzf /content/carla.tar.gz -C /content/carla && ls /content/carla

## 2. A Python the CARLA client supports
The 0.9.15 client wheels are for Python 3.7-3.10 (I believe). Colab's default Python is probably newer, so this cell
installs a separate Python 3.10 with conda. Could break: condacolab restarts the runtime once; re-run from the top of this
cell after the restart.

In [ ]:
!pip install -q condacolab
import condacolab; condacolab.install()          # the runtime restarts here

In [ ]:
!conda create -y -q -n carla310 python=3.10 numpy pydantic pyyaml
!/usr/local/envs/carla310/bin/pip install -q /content/carla/PythonAPI/carla/dist/carla-0.9.15-cp310-linux_x86_64.whl || /usr/local/envs/carla310/bin/pip install -q carla==0.9.15
!/usr/local/envs/carla310/bin/python -c "import carla, numpy; print('carla client ok')"

## 3. Start the server without a display
Our check needs only physics. Could break: no Vulkan on the Colab GPU. If the server dies, check `/content/carla.log`; then try
adding `-opengl`, or `-nullrhi` (no rendering at all; the collision sensor and vehicle physics should still work, but I have
not verified that for 0.9.15).

In [ ]:
import subprocess, time
log = open('/content/carla.log', 'w')
server = subprocess.Popen(['/content/carla/CarlaUE4.sh', '-RenderOffScreen', '-nosound', '-quality-level=Low'],
                          stdout=log, stderr=log)
time.sleep(60)
print('server alive' if server.poll() is None else 'server exited, see /content/carla.log')

In [ ]:
!tail -20 /content/carla.log

## 4. Run the check (a few minutes to an hour, depending on speed)

In [ ]:
!cd /content/proj && /usr/local/envs/carla310/bin/python experiments/e13_carla_plant.py --host localhost --port 2000 --out /content/e13_carla.json

In [ ]:
from google.colab import files
files.download('/content/e13_carla.json')       # send this file (and the printed table above) back